# Download and process WHACS offshore waves

Download WHACS bulk and partition spectra, remap them onto metamodel goals, and
write the processed zarr used by dynamic forcing and hindcast reconstruction.

**Prerequisites:** ``04_Select_Goals_by_Hindcasts.ipynb`` (``<Case>_Goals.json``).
**Next:** ``07b_Satellite_Calibration.ipynb`` validates (and calibrates) this
hindcast against satellite altimetry.

## Configuration


In [ ]:
import numpy as np
import xarray as xr

from utils.config import case_name, get_offshore_wave, load_goals, resolve

whacs_data = xr.open_dataset(
    resolve(f"inputs/WHACS/processed/{case_name()}_WHACS_seapoint.zarr")
).load()
whacs_data = whacs_data.assign({"tp": 1.0 / whacs_data["fp"]})
whacs_data["tp"].attrs = {
    "long_name": "Wave peak period",
    "units": "s",
}
whacs_data

In [ ]:
# Offshore node per goal, matched by lon/lat to the goals JSON snap points
# (hindcast.whacs.longitude/latitude). Some hindcasts are gridded rather
# than point-organized, so matching by position is more robust than by id.
goals = load_goals()
seapoint_lon = whacs_data["longitude"].isel(time=-1)
seapoint_lat = whacs_data["latitude"].isel(time=-1)
ordered_seapoints_for_goals = []
for goal in goals.values():
    snap = goal["hindcast"]["whacs"]
    distance = np.hypot(
        seapoint_lon - snap["longitude"], seapoint_lat - snap["latitude"]
    )
    ordered_seapoints_for_goals.append(
        int(whacs_data["seapoint"][int(distance.argmin())])
    )

whacs_data = whacs_data.sel(seapoint=ordered_seapoints_for_goals)[
    [
        "hs",
        "tp",
        "dir",
        "spr",
        "phs0",
        "ptp0",
        "pdir0",
        "pspr0",
        "phs1",
        "ptp1",
        "pdir1",
        "pspr1",
        "phs2",
        "ptp2",
        "pdir2",
        "pspr2",
        "phs3",
        "ptp3",
        "pdir3",
        "pspr3",
    ]
].rename({"seapoint": "goal"})
whacs_data["goal"] = [int(k) for k in goals]
whacs_data = whacs_data.assign_coords(
    {"seapoint": ("goal", ordered_seapoints_for_goals)}
)

Bulk `tp = 1 / fp` explodes when the spectral peak falls below the WHACS
frequency grid (hours with Tp of 50-250 s). Those take the peak period of the
most energetic partition (`clean_peak_periods`); partition periods above
`max_tp` become NaN.

In [ ]:
from bluemath_tk.waves.hywaves.offshore import clean_peak_periods, scale_partitions_to_bulk

max_tp = 25.0  # s, largest valid peak period
whacs_data = clean_peak_periods(whacs_data, max_tp=max_tp)
print(f"bulk tp replaced: {whacs_data.attrs['tp_replaced']}, partition tp removed: {whacs_data.attrs['ptp_removed']}")

**Partition energy.** WHACS keeps the wind sea and three swells; where the
sea state has more systems (multimodal tropical seas) the partitions hold less
energy than the bulk spectrum, and the spectral reconstruction (a sum of
partitions) would start low. `scale_partitions_to_bulk` gives each hour's
partitions the energy they miss, in proportion to theirs (`phs` x
`hs / sqrt(sum phs²)`; periods, directions and spreads kept). The share they
held before, per goal, is printed below. The satellite calibration (07b) scales
the bulk and the partitions by the same factors, so it keeps the closure.

In [ ]:
energy = sum(whacs_data[f"phs{i}"].fillna(0.0) ** 2 for i in range(4))
captured = (energy.sum("time") / (whacs_data["hs"] ** 2).sum("time")).to_series()
print("partition energy / bulk energy per goal, before scaling:")
print(captured.round(3).to_string())

whacs_data = scale_partitions_to_bulk(whacs_data)
whacs_data.to_zarr(resolve(get_offshore_wave("WHACS")["data_path"]), mode="w")

In [ ]:
xr.open_zarr(resolve(get_offshore_wave("WHACS")["data_path"])).load()